In [98]:
import sys
sys.path.append("../")

In [99]:
import torch
from qmpsqsc.models import mpsqsc
from qmpsqsc.models import qmps
from importlib import reload

reload(mpsqsc)

<module 'qmpsqsc.models.mpsqsc' from '/Users/keisuke/Documents/projects/mps4qsc/notebooks/../qmpsqsc/models/mpsqsc/__init__.py'>

In [100]:
import torch.nn.functional as F

L = 10
chi = 2
d = 2
dtype = torch.float64
qscr = mpsqsc.MpsQsc(L, chi, d, dtype=dtype, init="stacked")
allup = torch.zeros(L, d, dtype = dtype)
allup[:, 0] = 1.0
alldown = torch.zeros(L, d, dtype = dtype)
alldown[:, 1] = 1.0

mps_allup = mpsqsc.build_product_state(L, d, allup)
mps_alldown = mpsqsc.build_product_state(L, d, alldown)

mps_allup = mps_allup.normalize()
mps_alldown = mps_alldown.normalize()

ghz = mpsqsc.build_ghz_state(L, d, chi, dtype = torch.complex128)
ghz = ghz.normalize()

# qscr = qscr.canonicalize(truncate=True, normalize=True)

In [101]:
ghz = mpsqsc.build_ghz_state(L, d, chi)
ghz2 = mpsqsc.build_ghz_state(L, d, chi)
# Construct |000> - |111>
As = ghz2.As
As[0][:, 1] = -As[0][:, 1]
ghz2.set_As(As)
qsc = mpsqsc.build_2qsc_from_mpstate(ghz, ghz2)

for A in qsc.As:
    # add random noise to As
    A.data[:] += torch.randn_like(A) * 0.01
# qsc = qsc.truncate_bond_dimension(2)

# This classifier return 0 for ghz state and 50 / 50 for all up or all down

In [94]:
import random
from typing import List, Tuple, Dict
import torch

def _amps_to_probs_batch(amps: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    """
    Convert a batch of amplitudes to Born probabilities.
      amps: (B, 2) real or complex
      returns probs: (B, 2), each row sums to 1
    """
    if torch.is_complex(amps):
        abs_sq = (amps.conj() * amps).real
    else:
        abs_sq = amps * amps
    denom = abs_sq.sum(dim=-1, keepdim=True).clamp_min(eps)
    return abs_sq / denom

# -------------------------------------------------------------------------
# 1. function to create a dataset (batch)
# -------------------------------------------------------------------------

@torch.no_grad()
def create_ghz_rho_batch_qsc(
    mpsghz,
    mps_allup,
    mps_alldown,
    device,
    dtype,
    batch_size: int,
    error_rate: float,
) -> Tuple[List, torch.Tensor]:
    """
    Create one training batch for GHZ vs rho with local bit-flip errors.

    States composition:
        - 50% GHZ
        - 25% all-up
        - 25% all-down

    For each sample:
        1. Draw C ~ Poisson(L * error_rate) (L = number of sites).
        2. Choose C distinct sites uniformly at random.
        3. Flip the local tensor at those sites (Pauli-X in physical dimension).

    Returns:
        states: list of length batch_size, each an MPS-like object
        labels: LongTensor of shape (batch_size,) on `device`
                0 -> GHZ   (entangled)
                1 -> rho   (product states: all-up / all-down)
    """

    def _flip_sites_in_mps(mps_state, site_indices):
        """
        In-place flip of given sites in an MPS.

        Assumes:
            - mps_state[site] is a torch.Tensor
            - physical index is dimension 1 and has size 2
        """
        As = mps_state.As
        X = torch.tensor([[0, 1], [1, 0]], device=As[0].device, dtype=As[0].dtype)
        for i in site_indices:
            A = mps_state.As[i]
            if i != 0 and i != L - 1:
                A = torch.einsum("iaj, ab -> ibj", A, X)
            elif i == 0:
                A = torch.einsum("aj, ab -> bj", A, X)
            else:  # ind == L - 1
                A = torch.einsum("ia, ab -> ib", A, X)
            mps_state.As[i] = A

    # ---- Determine number of sites (assume all states have same length) ----
    num_sites = mpsghz.L

    # ---- Build the desired mixture: 50% GHZ, 25% all-up, 25% all-down ----
    num_ghz = batch_size // 2
    remaining = batch_size - num_ghz
    num_allup = remaining // 2
    num_alldown = remaining - num_allup

    # 0 = GHZ, 1 = all-up, 2 = all-down (internal codes)
    type_codes = (
        [0] * num_ghz +
        [1] * num_allup +
        [2] * num_alldown
    )
    type_codes = torch.tensor(type_codes, device=device)

    # Shuffle to avoid any ordering bias
    perm = torch.randperm(batch_size, device=device)
    type_codes = type_codes[perm]

    # ---- Sample number of errors per sample: C ~ Poisson(L * error_rate) ----
    # Treat error_rate as per-site error probability; total rate = L * error_rate
    states: List = []
    labels: List[int] = []

    for idx in range(batch_size):
        code = int(type_codes[idx].item())

        if code == 0:
            base_state = mpsghz
            label = 0  # GHZ
        elif code == 1:
            base_state = mps_allup
            label = 1  # product
        else:  # code == 2
            base_state = mps_alldown
            label = 1  # product

        state = base_state.copy()
        # Randomly generate 0 or 1 with probability error_rate for L times
        site_indices = [i for i in range(num_sites) if torch.rand(1).item() < error_rate]
        _flip_sites_in_mps(state, site_indices)
        states.append(state)
        labels.append(label)

    labels_tensor = torch.tensor(labels, dtype=torch.long, device=device)
    return states, labels_tensor

def robust_margin_loss(pred, y, M, eps=1e-6, K = 1000):
    """
    samples_zero: tensor (B, K) with 0/1 draws representing "model predicted class 0"
                  for each of K stochastic forward passes per input.
    y: LongTensor (B,) with ground truth labels 0 or 1.
    M: scalar threshold in probability space.
    """
    # Monte Carlo estimate of P0 and its SE
    mu = pred[:,0]
    var = mu * (1.0 - mu) / K
    se = (var + eps).sqrt()

    print(M)
    margin = (mu - M) / (se + eps)               # (B,)

    t = torch.where(y == 0, 1.0, -1.0)
    loss = torch.log1p(torch.exp(-t * margin))   # (B,)
    y_hat = torch.where(margin > 0,
                        torch.zeros_like(y),          # predict 0
                        torch.ones_like(y))           # predict 1

    acc = (y_hat == y).float().mean()

    return loss.mean(), acc

def qsc_ghz_rho_nll(
    qsc : mpsqsc.MpsQsc,
    states: List,
    labels: torch.Tensor,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Compute NLL loss and probabilities for GHZ vs ρ batch.

    Returns:
        loss: scalar tensor
        probs: (B, 2) tensor of Born probabilities
    """
    amps_list = [qsc.contract_with_state(st) for st in states]  # each (2,)
    amps_batch = torch.stack(amps_list, dim=0)                  # (B, 2)
    probs = _amps_to_probs_batch(amps_batch)                    # (B, 2)

    loss, acc = robust_margin_loss(probs, labels, qsc.D)
    return loss, probs, acc


# -------------------------------------------------------------------------
# 3. training loop using the two helpers above
# -------------------------------------------------------------------------

def train_classifier_qsc_on_ghz_vs_rho(
    qsc,
    mpsghz,
    mps_allup,
    mps_alldown,
    *,
    steps: int = 2000,
    lr: float = 1e-2,
    weight_decay: float = 0.0,
    grad_clip: float | None = None,
    log_every: int = 100,
    batch_size: int = 100,
    error_rate: float = 0.01,
) -> Dict[str, List[float]]:
    """
    Train qsc so that it outputs different classes for GHZ vs classical mixture ρ.
    Uses Born probabilities from 2-D amplitudes and NLL loss.
    """

    # if seed is not None:
    #     torch.manual_seed(seed)
    #     random.seed(seed)
    params = qsc.As + [qsc.D]
    optim = torch.optim.Adam(params, lr=lr, weight_decay=weight_decay)

    history: Dict[str, List[float]] = {"loss": [], "acc": []}

    for step in range(1, steps + 1):
        if hasattr(qsc, "train"):
            qsc.train()

        optim.zero_grad(set_to_none=True)

        # Create one batch (states + labels)
        states, labels = create_ghz_rho_batch_qsc(
            mpsghz, mps_allup, mps_alldown, qsc.device, qsc.dtype, batch_size, error_rate
        )

        # Forward + loss
        loss, probs, acc = qsc_ghz_rho_nll(qsc, states, labels)

        # Backward / step
        loss.backward()
        if grad_clip is not None and len(params) > 0:
            torch.nn.utils.clip_grad_norm_(params, max_norm=grad_clip)
        optim.step()


        history["loss"].append(float(loss.item()))
        history["acc"].append(acc)

        if (log_every is not None) and (step % log_every == 0):
            print(f"[step {step:5d}] loss={loss.item():.6f}  acc={acc:.3f}")

    return history


In [81]:
hist = train_classifier_qsc_on_ghz_vs_rho(
    qsc,
    mpsghz=ghz,
    mps_allup=mps_allup,
    mps_alldown=mps_alldown,
    steps=1000,
    lr=1e-3,
    weight_decay=0.0,
    grad_clip=1.0,   # optional
    log_every=1,
    batch_size=2**8,
    error_rate=0.08,
)

tensor(0.5783, dtype=torch.float64, requires_grad=True)
[step     1] loss=0.270637  acc=0.898
tensor(0.5773, dtype=torch.float64, requires_grad=True)
[step     2] loss=0.105522  acc=0.938


KeyboardInterrupt: 

In [116]:
Us = []

# add CNOT gate to Us for L - 1 times
for i in range(L - 1):
    Us.append(torch.tensor([[1, 0, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0], [0, 1, 0, 0]], dtype=torch.float64))

# last unitary is hadamard gate
last_unitary = torch.tensor([[1, 1], [1, -1]], dtype=torch.float64) / torch.sqrt(torch.tensor(2.0, dtype=torch.float64))
last_unitary =qmps.embed_non_unitary(last_unitary)


qmps_qsc = qmps.qMPS(L, d = d, chi = 2, Us=Us, last_unitary=last_unitary)

In [117]:
last_unitary

tensor([[ 7.0711e-01,  7.0711e-01,  1.4901e-08,  0.0000e+00],
        [ 7.0711e-01, -7.0711e-01,  0.0000e+00,  1.4901e-08],
        [ 1.4882e-08,  7.5161e-10, -7.0711e-01, -7.0711e-01],
        [ 7.5161e-10,  1.4882e-08, -7.0711e-01,  7.0711e-01]],
       dtype=torch.float64)

In [124]:
qmps_qsc._contract_circuit_with_state_partial_trace(mps_allup)

tensor([[0.5000, 0.5000],
        [0.5000, 0.5000]], dtype=torch.float64, grad_fn=<ViewBackward0>)